In [39]:
from __future__ import absolute_import, division, print_function
import tensorflow as tf
import numpy as np

In [38]:
# Setting Up Hyperparameters and Data Set Parameters

num_classes = 10 # 0 to 9 digits
num_features = 784 # 28*28
learning_rate = 0.01
training_steps = 1000
batch_size = 256
display_step = 50

In [34]:
# Loading and Preparing the MNIST Data Set

from tensorflow.keras.datasets import mnist

(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train, x_test = np.array(x_train, np.float32), np.array(x_test, np.float32)

x_train, x_test = x_train.reshape([-1, num_features]), x_test.reshape([-1, num_features])
x_train, x_test = x_train / 255., x_test / 255.

In [35]:
# Shuffling and Batching the Data

train_data=tf.data.Dataset.from_tensor_slices((x_train,y_train))

train_data=train_data.repeat().shuffle(5000).batch(batch_size).prefetch(1)

In [40]:
# Initializing Weights and Biases

W = tf.Variable(tf.ones([num_features, num_classes]), name="weight")
b = tf.Variable(tf.zeros([num_classes]), name="bias")

In [41]:
# Defining Logistic Regression and Cost Function

def logistic_regression(x):
    return tf.nn.softmax(tf.matmul(x, W) + b)

def cross_entropy(y_pred, y_true):
    y_true = tf.one_hot(y_true, depth=num_classes)
    y_pred = tf.clip_by_value(y_pred, 1e-9, 1.)
    return tf.reduce_mean(-tf.reduce_sum(y_true * tf.math.log(y_pred)))

In [36]:
# Defining Optimizers and Accuracy Metrics

def accuracy(y_pred, y_true):
    correct_prediction = tf.equal(tf.argmax(y_pred, 1), tf.cast(y_true, tf.int64))
    return tf.reduce_mean(tf.cast(correct_prediction, tf.float32))

optimizer = tf.optimizers.SGD(learning_rate)

In [37]:
# Optimization Process and Updating Weights and Biases

def run_optimization(x, y):
    with tf.GradientTape() as g:
        pred = logistic_regression(x)
        loss = cross_entropy(pred, y)
    gradients = g.gradient(loss, [W, b])
    optimizer.apply_gradients(zip(gradients, [W, b]))

In [42]:
# The Training Loop

for step, (batch_x, batch_y) in enumerate(train_data.take(training_steps), 1):
    run_optimization(batch_x, batch_y)
    if step % display_step == 0:
        pred = logistic_regression(batch_x)
        loss = cross_entropy(pred, batch_y)
        acc = accuracy(pred, batch_y)
        print("step: %i, loss: %f, accuracy: %f" % (step, loss, acc))

step: 50, loss: 1358.565918, accuracy: 0.613281
step: 100, loss: 1307.188843, accuracy: 0.703125
step: 150, loss: 1400.055176, accuracy: 0.644531
step: 200, loss: 1011.354858, accuracy: 0.734375
step: 250, loss: 1184.976807, accuracy: 0.742188
step: 300, loss: 1076.247070, accuracy: 0.769531
step: 350, loss: 1085.922363, accuracy: 0.734375
step: 400, loss: 1434.451538, accuracy: 0.664062
step: 450, loss: 1130.649902, accuracy: 0.750000
step: 500, loss: 831.784241, accuracy: 0.796875
step: 550, loss: 992.309387, accuracy: 0.765625
step: 600, loss: 1290.794067, accuracy: 0.710938
step: 650, loss: 1078.760010, accuracy: 0.746094
step: 700, loss: 1071.622803, accuracy: 0.773438
step: 750, loss: 1143.515015, accuracy: 0.710938
step: 800, loss: 1015.097778, accuracy: 0.707031
step: 850, loss: 1306.342896, accuracy: 0.726562
step: 900, loss: 942.187744, accuracy: 0.796875
step: 950, loss: 1225.602783, accuracy: 0.742188
step: 1000, loss: 1043.250000, accuracy: 0.773438


In [32]:
# Testing Model Accuracy Using the Test Data

pred = logistic_regression(x_test)
print("Test Accuracy: %f" % accuracy(pred, y_test))

Test Accuracy: 0.877400
